# FASE 3 (v2 - DIPERBARUI): YOLO26n + Coordinate Attention + Edge Contour (Google Colab GPU T4)
## Arsitektur Asli YOLO26n (`C3k2` + `SPPF` + `C2PSA` + `end2end=True`) + 100% Weight Remapping

Notebook ini menjalankan pipeline baru di folder **`phase3_yolo_ca/`** yang telah diperbaiki:
1. **Arsitektur Asli YOLO26n**: Menggunakan `C3k2`, `SPPF`, `C2PSA`, `reg_max=1`, dan `end2end=True` (bukan YOLOv8n `C2f`).
2. **100% Pretrained Weight Remapping**: Memetakan seluruh layer `yolo26n.pt` (`0..23` $\to$ `0..26`) sehingga 100% fitur COCO *backbone & neck* ditransfer.
3. **TFLite/ONNX Friendly Coordinate Attention**: Menggunakan `torch.mean(..., keepdim=True)` agar siap diekspor ke aplikasi Mobile.
4. **Perbandingan Fair (640×640 vs 640×640)**: Membandingkan `YOLO26n + CA (640×640)` terhadap `YOLO26n Baseline (640×640)`.
5. **Bebas Bug pada `06_test_external_image.py`**: Menguji gambar eksternal dengan visualisasi 4-panel tanpa *ImportError/AttributeError*.

### Langkah 1: Pastikan GPU T4 Aktif & Mount Google Drive

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

### Langkah 2: Siapkan Dataset (`tea_yolo.zip`) & Kode (`phase3_yolo_ca`)
Mendukung pengambilan kode otomatis dari **GitHub** (jika sudah di-*push*) maupun dari **Google Drive (`phase3_yolo_ca.zip`)**.

In [ ]:
import os

# 1. Ekstrak Dataset tea_yolo.zip dari Google Drive ke SSD lokal Colab (/content/)
if not os.path.exists('/content/tea_yolo'):
    !cp /content/drive/MyDrive/SKRIPSI/tea_yolo.zip /content/
    !unzip -q /content/tea_yolo.zip -d /content/
    print('[OK] Dataset tea_yolo berhasil diekstrak ke /content/tea_yolo')

# 2. Ambil kode phase3_yolo_ca & yolo26n.pt (dari Drive ZIP atau GitHub Clone)
if os.path.exists('/content/drive/MyDrive/SKRIPSI/phase3_yolo_ca.zip'):
    !cp /content/drive/MyDrive/SKRIPSI/phase3_yolo_ca.zip /content/
    !unzip -o -q /content/phase3_yolo_ca.zip -d /content/
    print('[OK] Kode phase3_yolo_ca diekstrak dari Google Drive!')
else:
    !git clone https://github.com/Kagalme/skripsi-tea-leaf-detection.git /content/repo_skripsi 2>/dev/null || (cd /content/repo_skripsi && git pull)
    !cp -r /content/repo_skripsi/phase3_yolo_ca /content/
    !cp /content/repo_skripsi/model_selection/yolo26n.pt /content/ 2>/dev/null || true
    print('[OK] Kode phase3_yolo_ca diambil dari GitHub!')

if not os.path.exists('/content/yolo26n.pt') and os.path.exists('/content/drive/MyDrive/SKRIPSI/yolo26n.pt'):
    !cp /content/drive/MyDrive/SKRIPSI/yolo26n.pt /content/

!ls -la /content/phase3_yolo_ca/
!ls -lh /content/yolo26n.pt

### Langkah 3: Install Dependency & Jalankan Unit Test Modul Baru

In [ ]:
!pip install -q ultralytics opencv-python-headless

!python /content/phase3_yolo_ca/ca_module.py
!python /content/phase3_yolo_ca/edge_contour.py

### Langkah 4: Training YOLO26n + Coordinate Attention (50 Epoch, 640×640, Batch 16)
Otomatis memetakan 100% bobot `yolo26n.pt` ke layer `yolo26n_ca.yaml` sebelum memulai training (~8–12 menit di GPU T4).

In [ ]:
!python /content/phase3_yolo_ca/01_train_ca.py \
    --device cuda \
    --epochs 50 \
    --imgsz 640 \
    --batch 16 \
    --workers 8 \
    --run_name yolo26n_ca_50ep

### Langkah 5: Evaluasi Test Set (529 Gambar, 640×640) & Visual Analysis (TP, FP, WC, FN)

In [ ]:
!python /content/phase3_yolo_ca/02_evaluate_testset.py \
    --device cuda \
    --imgsz 640 \
    --batch 16 \
    --workers 8 \
    --run_name yolo26n_ca_50ep

!python /content/phase3_yolo_ca/03_visual_analysis.py \
    --device cuda \
    --imgsz 640 \
    --run_name yolo26n_ca_50ep \
    --conf 0.25 \
    --iou_thr 0.5 \
    --max_samples 10

### Langkah 6: Perbandingan Adil (640×640): YOLO26n Baseline vs YOLO26n + CA

In [ ]:
!python /content/phase3_yolo_ca/04_compare_baseline_vs_ca.py \
    --run_name_ca yolo26n_ca_50ep \
    --run_name_baseline yolo26n

from IPython.display import display, Markdown
with open('/content/laporan/fase3_ca_vs_baseline.md', 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))

### Langkah 7: Inferensi Edge Contour & Filter Daun Tidak Utuh pada Test Set

In [ ]:
!python /content/phase3_yolo_ca/05_infer_with_contour.py \
    --source /content/tea_yolo/images/test \
    --weights /content/phase3_runs/yolo26n_ca_50ep/weights/best.pt \
    --device cuda \
    --conf 0.25 \
    --max_images 20 \
    --save \
    --save_stats \
    --show_suppressed

import glob
import matplotlib.pyplot as plt
from PIL import Image

output_images = sorted(glob.glob('/content/phase3_runs/contour_inference/*.jpg'))[:6]
if output_images:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, p in zip(axes.flatten(), output_images):
        ax.imshow(Image.open(p))
        ax.set_title(p.split('/')[-1], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

### Langkah 8: Uji Gambar Eksternal / Upload Foto Daun Sendiri (4-Panel Visualisasi)

In [ ]:
import glob
from IPython.display import Image as IPyImage, display

# Contoh uji pada salah satu gambar test atau ganti dengan --url / file upload
sample_img = sorted(glob.glob('/content/tea_yolo/images/test/*.jpg'))[0]
!python /content/phase3_yolo_ca/06_test_external_image.py \
    --image "$sample_img" \
    --weights /content/phase3_runs/yolo26n_ca_50ep/weights/best.pt \
    --conf 0.25

display(IPyImage('test_google_results/hasil_testing_lengkap.png'))

### Langkah 9: Cadangkan Seluruh Hasil & Laporan ke Google Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/SKRIPSI/phase3_runs/yolo26n_ca_50ep
!mkdir -p /content/drive/MyDrive/SKRIPSI/phase3_runs/contour_inference
!mkdir -p /content/drive/MyDrive/SKRIPSI/phase3_runs/test_google_results
!mkdir -p /content/drive/MyDrive/SKRIPSI/laporan

!cp -r /content/phase3_runs/yolo26n_ca_50ep/* /content/drive/MyDrive/SKRIPSI/phase3_runs/yolo26n_ca_50ep/
!cp -r /content/phase3_runs/contour_inference/* /content/drive/MyDrive/SKRIPSI/phase3_runs/contour_inference/ 2>/dev/null || true
!cp -r test_google_results/* /content/drive/MyDrive/SKRIPSI/phase3_runs/test_google_results/ 2>/dev/null || true
!cp /content/laporan/fase3_ca_vs_baseline.md /content/drive/MyDrive/SKRIPSI/laporan/

print('[SELESAI] Semua bobot best.pt, grafik, dan laporan Fase 3 telah tersimpan di Google Drive!')